In [2]:
#setup cell, imports tools
#loads JPL files
#test-pulls Earth's position at ASSIST's starting time
import numpy as np
import rebound #gravity sim
import assist #JPL addon
from astroquery.jplhorizons import Horizons #asks for official position
from pathlib import Path

DATA = Path.home() / "assist_data" #saves location of data folder
ephem = assist.Ephem(str(DATA / "linux_p1550p2650.440"), str(DATA / "sb441-n16.bsp")) #opens JPL files, loads them into ASSIST. 
print(ephem.jd_ref) #prints ASSIST's start as a Julian Date
print(ephem.get_particle("Earth", 0)) #asks where earth was at start date

2451545.0
<rebound.particle.Particle object at 0x71d1f4987750, m=8.887692446707103e-10 x=-0.18427227843428148 y=0.8847811839446426 z=0.3838199903352723 vx=-0.017202246608380413 vy=-0.002904925902856414 vz=-0.0012594279121938968>


In [5]:
#This cell will test the program with Eros. It will ask for its position on two dates.
T0 = 2460919.5 #midnight, 09/01/2025, the start date
T1 = T0 + 60 #60 days later, the end date

eros = Horizons(id="433", id_type="smallbody", location="@ssb", epochs=[T0, T1]).vectors(refplane="earth") #asks for Eros's positon and speed
#id="433" is Eros's asteroid ID number
#id_type="smallbody" tells horizions what 433 means, that it's an asteroid number
#location = "@ssb" is the measure from the barycenter 
#epochs=[T0, T1] are the two dates I want it to give me info for
#.vectors(...) asks for position and speed instead of sky coords
#refplane="earth" lines up the x, y, and z with earth's equators. This prevents Horizons from tilting everything to line up with Earth's orbit
eros["datetime_str", "x", "y", "z", "vx", "vy", "vz"] #shows date and six numbers exclusively instead of every output horizon gives back

datetime_str,x,y,z,vx,vy,vz
---,AU,AU,AU,AU / d,AU / d,AU / d
str30,float64,float64,float64,float64,float64,float64
A.D. 2025-Sep-01 00:00:00.0000,1.525008891559722,-0.01175486543019075,0.2704847339134051,-0.003865857498908609,0.01139340864437072,0.005768960813105879
A.D. 2025-Oct-31 00:00:00.0000,1.066833277500576,0.6345149904363366,0.5545379227820589,-0.01145525145830082,0.009401972080356077,0.003266573051507908


In [11]:
#this cell's purpose is to test the propagtor by letting it push an objet (Eros), then testing it against JPL's actual value
sim = rebound.Simulation() #makes new simulation
sim.add(x=eros["x"][0], y=eros["y"][0], z=eros["z"][0], vx=eros["vx"][0], vy=eros["vy"][0], vz=eros["vz"][0]) #puts eros in the simulation using day 0
sim.t = T0 - ephem.jd_ref #sets clock
sim.ri_ias15.min_dt = 0.001 #stops REBOUND from taking steps smaller than 0.001, so as to not waste time (follows ASSIST's own example)
extras = assist.Extras(sim, ephem) #attaches ASSIST, aso the largest bodies int eh solar system pull on Eros
extras.gr_eih_sources = 11 #turns on Einstien's relativity correction.

sim.integrate(T1 - ephem.jd_ref) #runs simulation forward to day 60
p = sim.particles[0] #grabs Eros's position
miss_km = np.sqrt((p.x - eros["x"][1])**2 + (p.y - eros["y"][1])**2 + (p.z - eros["z"][1])**2) * 149597870.7 #3D distance formula, converts au to km
print(f"Eros: My propogator is {miss_km * 1000:.3f} m off from JPL after 60 days") #prints miss

Eros: My propogator is 0.190 m off from JPL after 60 days


In [12]:
#this cell will run the same test as the one above, but for 3I
iso = Horizons(id="C/2025 N1", id_type="designation", location="@ssb", epochs=[T0, T1]).vectors(refplane="earth") #gets 1I's position and speed, but the id is for a comet ("designation")

sim = rebound.Simulation() #Tis line, and the below five do the exact same as the cell above, so look there for comments
sim.add(x=iso["x"][0], y=iso["y"][0], z=iso["z"][0], vx=iso["vx"][0], vy=iso["vy"][0], vz=iso["vz"][0])
sim.t = T0 - ephem.jd_ref
sim.ri_ias15.min_dt = 0.001
extras = assist.Extras(sim, ephem)
extras.gr_eih_sources = 11

sim.integrate(T1 - ephem.jd_ref) #This cell and the next two below are also the same as in the cell above
p = sim.particles[0]
miss_km = np.sqrt((p.x - iso["x"][1])**2 + (p.y - iso["y"][1])**2 + (p.z - iso["z"][1])**2) * 149597870.7
print(f"3I: my gravity-only propogator is {miss_km:.1f} km off from JPL after 60 days") #only 1 decimal point bc of larger expected result

3I: my gravity-only propogator is 1778.2 km off from JPL after 60 days
